In [6]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA


from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, roc_auc_score


In [7]:
df = pd.read_parquet("../data/processed/dataset_features.parquet")

df = df[df["condition"].isin(["EO", "EC"])]

df["y"] = df["condition"].map({"EO": 0, "EC": 1})

In [8]:
NODE_FEATURES = [
    c for c in df.columns
    if c.startswith(("degree_", "clustering_", "betweenness_", "hub_"))
]

In [9]:
META_FEATURES = ["age", "gender"]

In [14]:
def evaluate_model(X, y, model):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    scoring = {
        "acc": "accuracy",
        "auc": "roc_auc"
    }

    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring
    )

    acc_mean = scores["test_acc"].mean()
    acc_std = scores["test_acc"].std()

    auc_mean = scores["test_auc"].mean()
    auc_std = scores["test_auc"].std()

    print(
        "=== Model Evaluation (Stratified 5-Fold CV) ===\n"
        f"Mean Accuracy : {acc_mean:.3f} ± {acc_std:.3f}\n"
        f"Mean ROC-AUC  : {auc_mean:.3f} ± {auc_std:.3f}\n"
        "Note: ROC-AUC values close to 0.5 indicate random performance."
    )

    return {
        "accuracy_mean": acc_mean,
        "auc_mean": auc_mean
    }


A  função acima treina e avalia um modelo de classificação usando validação cruzada estratificada com 5 folds, calcula acurácia e AUC em cada fold e retorna a média dessas métricas.

### Regressão Logística

In [15]:
X = df[NODE_FEATURES].select_dtypes(include="number").dropna(axis=1)
y = df["y"]

pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000))
])

evaluate_model(X, y, pipe_lr)


=== Model Evaluation (Stratified 5-Fold CV) ===
Mean Accuracy : 0.421 ± 0.024
Mean ROC-AUC  : 0.422 ± 0.064
Note: ROC-AUC values close to 0.5 indicate random performance.


{'accuracy_mean': np.float64(0.4208333333333333),
 'auc_mean': np.float64(0.4222886029411764)}

Avaliação do modelo por validação cruzada estratificada (5 folds).
O modelo apresentou acurácia média de 42,1% e AUC média de 0,42, indicando um desempenho inferior ao esperado para um classificador binário.

A acurácia próxima a 50% sugere que o modelo possui baixa capacidade preditiva, enquanto o valor de AUC abaixo de 0,5 indica que o modelo não consegue separar adequadamente as classes, apresentando desempenho próximo — ou até inferior — ao de um classificador aleatório.

Esses resultados indicam a necessidade de revisão das features, engenharia de atributos, ajuste de hiperparâmetros ou a avaliação de modelos mais complexos.

### Regressão Logística com PCA

In [16]:

pipe_pca_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=10)),
    ("clf", LogisticRegression(max_iter=1000))
])

evaluate_model(X, y, pipe_pca_lr)


=== Model Evaluation (Stratified 5-Fold CV) ===
Mean Accuracy : 0.463 ± 0.059
Mean ROC-AUC  : 0.453 ± 0.087
Note: ROC-AUC values close to 0.5 indicate random performance.


{'accuracy_mean': np.float64(0.46344696969696975),
 'auc_mean': np.float64(0.4530790441176471)}

### Regressão Logística com PCA e metadados

In [17]:
X_meta = pd.concat(
    [X, df[META_FEATURES]],
    axis=1
).dropna()

y_meta = df.loc[X_meta.index, "y"]

pipe_pca_meta = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=10)),
    ("clf", LogisticRegression(max_iter=1000))
])

evaluate_model(X_meta, y_meta, pipe_pca_meta)


KeyError: "None of [Index(['age', 'gender'], dtype='object')] are in the [columns]"